# Практическая работа 3. Построение сквозного ML-пайплайна на больших данных с помощью Spark MLlib

**Студент:** Войт Иван Иванович  
**Группа:** БД-251М  
**Вариант:** 10

## Тема моего варианта

В этой работе я решаю задачу бинарной классификации: по характеристикам тренировки предсказываю `gender` пользователя.

Для моего варианта нужно:

1. Построить признаки `avg_speed` и `stddev_speed`.
2. Обучить модель `LogisticRegression` с регуляризацией `ElasticNet`.
3. Интерпретировать коэффициенты модели и объяснить, как стабильность темпа связана с полом пользователя.

## Логика работы ноутбука

Я разбил решение на логические этапы, как того требуют методические указания:

- `Setup` — подготовка среды в Google Colab;
- `ETL` — загрузка и очистка данных;
- `Feature Engineering` — вычисление признаков из массивов;
- `ML Pipeline` — построение конвейера `pyspark.ml.Pipeline`;
- `Evaluation` — расчет метрик качества;
- `Business Interpretation` — перевод результата в бизнес-смысл.

Ниже я стараюсь подробно комментировать почти каждый шаг, чтобы было видно не только **что** я делаю, но и **зачем**.

## 0. Setup — подготовка окружения

В этой части я устанавливаю и импортирую библиотеки, которые понадобятся мне для работы со Spark, визуализации и оценки модели.

Я специально использую `pyspark==3.4.1`, потому что эта версия хорошо подходит под формат лабораторной работы и совместима с типовым стеком Spark MLlib.

In [ ]:
import sys

# В Colab иногда уже есть пакет dataproc-spark-connect, который конфликтует с нужной мне
# версией PySpark. Поэтому я сначала удаляю его, чтобы установка прошла чище.
!{sys.executable} -m pip uninstall -y dataproc-spark-connect >/dev/null 2>&1

# Устанавливаю все зависимости, которые нужны для Spark ML pipeline и визуализации.
!{sys.executable} -m pip install pyspark==3.4.1 findspark matplotlib seaborn scikit-learn pandas numpy --quiet

print('Зависимости установлены')

In [ ]:
import os
import shutil
import warnings
warnings.filterwarnings('ignore')

import findspark
findspark.init()

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix, roc_curve, auc

print(f'PySpark version: {pyspark.__version__}')

## 1. Загрузка данных

Исходный файл `endomondoHR.json` очень большой, поэтому для работы в Colab я использую уменьшенную версию набора данных.

В этой ячейке я делаю код удобным для запуска:

- сначала проверяю, есть ли файл в папке `data/`;
- если файла нет, прошу загрузить его через интерфейс Colab;
- после загрузки сохраняю его в `data/`, чтобы дальше код работал одинаково.

Это делает тетрадь самодостаточной: её можно запустить даже в чистой сессии Colab.

In [ ]:
# Создаю каталог data, если его еще нет.
os.makedirs('data', exist_ok=True)

# Здесь перечисляю возможные пути, по которым может лежать файл.
# Я учитываю как вариант с папкой data, так и ручную загрузку прямо в корень сессии Colab.
DATA_CANDIDATES = [
    'data/endomondoHR_sample_v10.json.gz',
    'data/endomondoHR_sample_v10.json',
    'endomondoHR_sample_v10.json.gz',
    'endomondoHR_sample_v10.json',
    'data/endomondoHR.json',
    'endomondoHR.json'
]

# Ищу первый реально существующий файл.
DATA_PATH = next((p for p in DATA_CANDIDATES if os.path.exists(p)), None)

if DATA_PATH is None:
    # Если файл не найден, пробую использовать стандартный механизм загрузки Colab.
    from google.colab import files

    print('Файл не найден автоматически.')
    print('Пожалуйста, загрузите один из файлов:')
    print('- endomondoHR_sample_v10.json.gz')
    print('- endomondoHR_sample_v10.json')
    print('- endomondoHR.json')

    uploaded = files.upload()

    if not uploaded:
        raise FileNotFoundError('Файл для анализа не был загружен в сессию Colab.')

    uploaded_name = next(iter(uploaded))
    target_path = os.path.join('data', uploaded_name)

    # Перемещаю файл в папку data, чтобы дальше все этапы использовали единый путь.
    if uploaded_name != target_path:
        shutil.move(uploaded_name, target_path)

    DATA_PATH = target_path

print('Используемый файл:', DATA_PATH)

## 2. Создание SparkSession

Теперь я инициализирую Spark. Это основной объект, через который выполняются все операции чтения, преобразования и обучения моделей в PySpark.

Я также уменьшаю число `shuffle partitions`, чтобы работа в Colab была немного легче и быстрее.

In [ ]:
# Создаю SparkSession для выполнения всех дальнейших шагов.
spark = (
    SparkSession.builder
    .appName('HealthTech_Gender_Predictor_V10')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '32')
    .getOrCreate()
)

# Отключаю лишние служебные сообщения, чтобы вывод был чище.
spark.sparkContext.setLogLevel('ERROR')

print('Spark version:', spark.version)
print('App name:', spark.sparkContext.appName)

In [ ]:
# Считываю JSON-файл в Spark DataFrame.
raw_df = spark.read.json(DATA_PATH)

print('Схема исходных данных:')
raw_df.printSchema()

# Показываю количество записей и несколько строк для первичного знакомства с данными.
raw_count = raw_df.count()
print(f'Количество сырых записей: {raw_count:,}')
raw_df.show(3, truncate=80)

## 3. ETL — первичный анализ и очистка данных

На этом этапе я проверяю качество данных и оставляю только те записи, которые действительно подходят для решения моей задачи.

Почему очистка важна:

- логистическая регрессия не сможет качественно работать, если в признаках много пропусков;
- признак `stddev_speed` невозможно посчитать без массива `speed`;
- слишком короткие массивы скорости слабо отражают реальный темп тренировки.

Поэтому я заранее отфильтровываю неподходящие строки.

In [ ]:
# Смотрю, сколько пропусков есть в наиболее важных для моей задачи полях.
print('Пропуски по ключевым полям:')
for c in ['gender', 'sport', 'speed', 'heart_rate']:
    null_cnt = raw_df.filter(F.col(c).isNull()).count()
    print(f'{c:12s}: {null_cnt:>8,}')

print('\nРаспределение gender:')
raw_df.groupBy('gender').count().orderBy(F.desc('count')).show()

print('Топ-10 sports:')
raw_df.groupBy('sport').count().orderBy(F.desc('count')).show(10, truncate=False)

In [ ]:
# Оставляю только строки с известным полом male/female.
# Вариант unknown убираю, потому что он не подходит для бинарной классификации.
# Также оставляю только те записи, где speed существует и содержит хотя бы 10 точек.
df_clean = (
    raw_df
    .filter(F.col('gender').isin('male', 'female'))
    .filter(F.col('speed').isNotNull())
    .filter(F.size(F.col('speed')) >= 10)
)

clean_count = df_clean.count()
print(f'Количество записей после очистки: {clean_count:,}')
print(f'Удалено записей: {raw_count - clean_count:,}')

df_clean.groupBy('gender').count().orderBy(F.desc('count')).show()

## 4. Feature Engineering — построение признаков

Это ключевая часть моего варианта.

Я создаю следующие признаки:

- `avg_speed` — средняя скорость тренировки;
- `stddev_speed` — стандартное отклонение скорости, которое показывает, насколько темп был стабильным;
- `speed_len` — длина массива скорости, то есть сколько измерений есть в тренировке.

### Что означают эти признаки

- `avg_speed` показывает общий уровень интенсивности движения;
- `stddev_speed` отражает вариативность темпа: чем он выше, тем менее ровной была скорость;
- `speed_len` косвенно характеризует продолжительность и детализацию записи.

Так как поля в JSON представлены массивами, я вычисляю статистики прямо средствами Spark SQL.

In [ ]:
# Формула для среднего значения массива speed.
# Здесь я суммирую все элементы массива и делю на его длину.
mean_expr = "aggregate(speed, cast(0.0 as double), (acc, x) -> acc + x) / size(speed)"

# Формула для дисперсии. Сначала для каждого элемента массива считаю квадрат отклонения
# от среднего, затем суммирую эти значения и делю на n - 1.
# После этого ниже беру квадратный корень и получаю стандартное отклонение.
var_expr = f"""
aggregate(
    transform(speed, x -> pow(x - ({mean_expr}), 2)),
    cast(0.0 as double),
    (acc, x) -> acc + x
) / (size(speed) - 1)
"""

# Формирую DataFrame уже с готовыми признаками.
df_features = (
    df_clean
    .select(
        'gender',
        'sport',
        F.size('speed').alias('speed_len'),
        F.expr(mean_expr).alias('avg_speed'),
        F.when(
            F.size('speed') > 1,
            F.sqrt(F.expr(var_expr))
        ).otherwise(F.lit(0.0)).alias('stddev_speed')
    )
    .dropna(subset=['avg_speed', 'stddev_speed'])
)

# Список признаков сохраняю отдельно, чтобы потом использовать его в VectorAssembler.
FEATURE_COLS = ['avg_speed', 'stddev_speed', 'speed_len']

df_features.show(5, truncate=False)
df_features.select(
    F.mean('avg_speed').alias('mean_avg_speed'),
    F.mean('stddev_speed').alias('mean_stddev_speed'),
    F.mean('speed_len').alias('mean_speed_len')
).show()

In [ ]:
# Для наглядности перевожу часть данных в pandas и строю простые распределения признаков.
# Это помогает понять, есть ли визуальная разница между male и female еще до обучения модели.
sample_pd = df_features.select('gender', 'avg_speed', 'stddev_speed', 'speed_len').sample(False, 0.25, seed=42).toPandas()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(data=sample_pd, x='avg_speed', hue='gender', kde=True, ax=axes[0], element='step')
axes[0].set_title('Распределение avg_speed')

sns.histplot(data=sample_pd, x='stddev_speed', hue='gender', kde=True, ax=axes[1], element='step')
axes[1].set_title('Распределение stddev_speed')

sns.histplot(data=sample_pd, x='speed_len', hue='gender', kde=True, ax=axes[2], element='step')
axes[2].set_title('Распределение speed_len')

plt.tight_layout()
plt.show()

## 5. Построение ML Pipeline

Теперь я перехожу к этапу машинного обучения.

Я использую именно `pyspark.ml.Pipeline`, потому что это одно из требований задания и хороший промышленный подход.

В мой pipeline входят следующие этапы:

1. `StringIndexer` — перевожу строковый `gender` в числовой `label`.
2. `VectorAssembler` — объединяю признаки в единый вектор `features`.
3. `StandardScaler` — масштабирую признаки.
4. `LogisticRegression` — обучаю модель с ElasticNet-регуляризацией.

### Почему нужен StandardScaler

Логистическая регрессия чувствительна к масштабу признаков. Если один признак измеряется в очень больших значениях, а другой в маленьких, это может повлиять на устойчивость обучения. Поэтому перед моделью я нормализую признаки.

In [ ]:
# Чтобы классы в train и test были представлены более равномерно,
# я делаю раздельное разбиение для male и female, а затем объединяю результаты.
female_df = df_features.filter(F.col('gender') == 'female')
male_df = df_features.filter(F.col('gender') == 'male')

female_train, female_test = female_df.randomSplit([0.8, 0.2], seed=42)
male_train, male_test = male_df.randomSplit([0.8, 0.2], seed=42)

train_df = female_train.unionByName(male_train)
test_df = female_test.unionByName(male_test)

print(f'Размер train: {train_df.count():,}')
print(f'Размер test : {test_df.count():,}')

print('Распределение классов в train:')
train_df.groupBy('gender').count().orderBy(F.desc('count')).show()

print('Распределение классов в test:')
test_df.groupBy('gender').count().orderBy(F.desc('count')).show()

In [ ]:
# Кодирую target-переменную gender в числовой label.
label_indexer = StringIndexer(inputCol='gender', outputCol='label', handleInvalid='skip')

# Собираю числовые признаки в единый вектор features.
assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol='features')

# Масштабирую признаки перед обучением логистической регрессии.
scaler = StandardScaler(inputCol='features', outputCol='scaled_features', withMean=True, withStd=True)

# Создаю модель LogisticRegression.
# regParam отвечает за силу регуляризации.
# elasticNetParam=0.5 означает смесь L1 и L2 регуляризации.
lr = LogisticRegression(
    labelCol='label',
    featuresCol='scaled_features',
    maxIter=40,
    regParam=0.15,
    elasticNetParam=0.5
)

# Объединяю все этапы в единый pipeline.
pipeline = Pipeline(stages=[label_indexer, assembler, scaler, lr])

# Обучаю pipeline на тренировочных данных.
pipeline_model = pipeline.fit(train_df)

# Применяю готовую модель к тестовым данным.
predictions = pipeline_model.transform(test_df)

predictions.select('gender', 'label', 'prediction', 'probability').show(10, truncate=False)

## 6. Оценка качества модели

На этом этапе я проверяю, насколько хорошо модель справилась с классификацией.

Я использую:

- `ROC-AUC` — показывает, насколько хорошо модель разделяет два класса в целом;
- `Accuracy` — показывает долю правильно классифицированных объектов;
- `Confusion Matrix` — помогает увидеть, какие ошибки модель делает чаще.

Такой набор метрик дает и общее, и более прикладное понимание качества.

In [ ]:
# Evaluator для ROC-AUC.
roc_eval = BinaryClassificationEvaluator(
    labelCol='label',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderROC'
)

# Evaluator для accuracy.
acc_eval = MulticlassClassificationEvaluator(
    labelCol='label',
    predictionCol='prediction',
    metricName='accuracy'
)

roc_auc = roc_eval.evaluate(predictions)
accuracy = acc_eval.evaluate(predictions)

# Перевожу часть результатов в pandas, чтобы построить ROC-кривую и confusion matrix.
preds_pd = predictions.select('label', 'prediction', 'probability').toPandas()
preds_pd['prob_1'] = preds_pd['probability'].apply(lambda v: float(v[1]))

cm = confusion_matrix(preds_pd['label'], preds_pd['prediction'])
fpr, tpr, _ = roc_curve(preds_pd['label'], preds_pd['prob_1'])
roc_curve_auc = auc(fpr, tpr)

print(f'ROC-AUC : {roc_auc:.4f}')
print(f'Accuracy: {accuracy:.4f}')
print('Confusion Matrix:')
print(cm)

In [ ]:
# Создаю папку для графиков, чтобы результаты можно было сохранять в репозиторий или отчет.
os.makedirs('figures', exist_ok=True)
sns.set_theme(style='whitegrid')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC-кривая показывает компромисс между true positive rate и false positive rate.
axes[0].plot(fpr, tpr, label=f'ROC AUC = {roc_curve_auc:.3f}', color='#1f77b4', linewidth=2)
axes[0].plot([0, 1], [0, 1], '--', color='gray')
axes[0].set_title('ROC-кривая')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()

# Матрица ошибок наглядно показывает, сколько объектов каждого класса модель угадала и перепутала.
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title('Матрица ошибок')
axes[1].set_xlabel('Predicted label')
axes[1].set_ylabel('True label')

plt.tight_layout()
plt.savefig('figures/fig_v10_roc_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Интерпретация коэффициентов логистической регрессии

Одно из преимуществ логистической регрессии состоит в том, что её можно интерпретировать.

Я извлекаю коэффициенты модели и смотрю:

- какой признак сильнее влияет на классификацию;
- в какую сторону действует каждый признак;
- насколько важен `stddev_speed`, то есть стабильность темпа.

### Как читать коэффициенты

- положительный коэффициент: рост признака увеличивает вероятность положительного класса;
- отрицательный коэффициент: рост признака уменьшает вероятность положительного класса;
- коэффициент около нуля: признак почти не влияет на решение модели.

In [ ]:
# Извлекаю обученный StringIndexer и саму логистическую регрессию из pipeline.
label_model = pipeline_model.stages[0]
lr_model = pipeline_model.stages[-1]

# labels показывает, какой строковый класс каким индексом закодирован.
label_order = label_model.labels
positive_class = label_order[1]

# Собираю коэффициенты в pandas DataFrame для удобного просмотра.
coef_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'coefficient': lr_model.coefficients.toArray()
}).sort_values('coefficient', ascending=False)

print('Порядок меток StringIndexer:', label_order)
print('Положительный класс модели:', positive_class)
print(f'Intercept: {lr_model.intercept:.4f}')
display(coef_df)

plt.figure(figsize=(8, 4))
sns.barplot(data=coef_df, x='coefficient', y='feature', palette='viridis')
plt.axvline(0, color='black', linewidth=1)
plt.title(f'Коэффициенты LogisticRegression для класса {positive_class}')
plt.tight_layout()
plt.savefig('figures/fig_v10_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Бизнес-интерпретация результата

Теперь я перевожу технический результат в бизнес-контекст.

Важно понимать, что такая модель не должна использоваться как жесткий способ определения пола пользователя. Гораздо корректнее рассматривать её как **дополнительный вероятностный сигнал** для:

- персонализации предложений;
- более точной сегментации пользователей;
- уточнения пользовательского профиля при неполных данных;
- повышения релевантности маркетинговых коммуникаций.

Если коэффициент `stddev_speed` заметен по модулю, это означает, что стабильность или нестабильность темпа действительно несет поведенческую информацию.

In [ ]:
# Перевожу коэффициенты в словарь, чтобы быстро обращаться к ним по имени признака.
coef_map = dict(zip(coef_df['feature'], coef_df['coefficient']))
std_coef = coef_map.get('stddev_speed', 0.0)

print('=' * 72)
print('ИТОГОВАЯ СВОДКА — ПР-3, Вариант 10')
print('=' * 72)
print(f'Используемый датасет      : {DATA_PATH}')
print(f'Записей после очистки     : {clean_count:,}')
print(f'Модель                    : LogisticRegression + ElasticNet')
print(f'Признаки                  : {FEATURE_COLS}')
print(f'ROC-AUC                   : {roc_auc:.4f}')
print(f'Accuracy                  : {accuracy:.4f}')
print(f'Положительный класс       : {positive_class}')
print(f'coef(avg_speed)           : {coef_map.get("avg_speed", 0.0):+.4f}')
print(f'coef(stddev_speed)        : {std_coef:+.4f}')
print(f'coef(speed_len)           : {coef_map.get("speed_len", 0.0):+.4f}')
print('-' * 72)

# Формирую текстовый вывод по признаку стабильности темпа.
if abs(std_coef) < 0.05:
    business_text = (
        'По моему результату стабильность темпа почти не влияет на определение пола. '
        'Значит, использовать только этот показатель как самостоятельный бизнес-сигнал не стоит.'
    )
elif std_coef > 0:
    business_text = (
        f'По моей модели рост stddev_speed повышает вероятность класса {positive_class}. '
        'Это означает, что более нестабильный темп связан с этим классом сильнее и может '
        'использоваться как дополнительный поведенческий сигнал в персонализации.'
    )
else:
    business_text = (
        f'По моей модели более стабильный темп сильнее связан с классом {positive_class}. '
        'Следовательно, равномерность скорости можно учитывать как мягкий признак при '
        'сегментации пользователей и настройке рекомендаций.'
    )

print(business_text)
print('=' * 72)

## 9. Краткий вывод по работе

В этой лабораторной работе я:

1. Загрузил и очистил данные Endomondo в среде Apache Spark.
2. Построил признаки `avg_speed`, `stddev_speed` и `speed_len` на основе массивов скорости.
3. Реализовал полный `pyspark.ml.Pipeline`.
4. Обучил модель `LogisticRegression` с регуляризацией `ElasticNet`.
5. Оценил качество модели по метрикам `ROC-AUC` и `Accuracy`.
6. Интерпретировал коэффициенты и сформулировал бизнес-вывод.

Таким образом, требования варианта 10 и общие требования к лабораторной работе выполнены.